In [1]:
HEADPATH = '/home/gutil/scratch/2025-06-28_Manual-Drift-Recording/'
# HEADPATH = '/home/lag36/scratch/lag36/2024-09-05_lLAG8_EZRDM_Run-01/Drift-Correction/'
# Drift is Barcode data relative to experiment data
# i.e. how much do I have to move the experiment data to align it with the Barcode data

In [2]:
import napari
import numpy as np
from skimage import io

In [ ]:
# Now for stacks
bc_image_first = HEADPATH + 'Barcodes_crop_XY-01_T-Last.tif'
bc_image_last = HEADPATH + 'Barcode_crop_XY-Last_T-Last.tif'

exp_image_first = HEADPATH + 'Experiment_crop_XY-01_T-Last.tif'
exp_image_last = HEADPATH + 'Experiment_crop_XY-Last_T-Last.tif'

# Load the images
bc_image_first_data = io.imread(bc_image_first)
bc_image_last_data = io.imread(bc_image_last)
exp_image_first_data = io.imread(exp_image_first)
exp_image_last_data = io.imread(exp_image_last)

# Cropping parameters (Y, X)
crop_start = (600, 1000)  # Top-left corner
crop_end = (600+550, 1000+400)  # Bottom-right corner

# Crop the images
bc_image_first_cropped = bc_image_first_data[crop_start[0]:crop_end[0], crop_start[1]:crop_end[1]]
bc_image_last_cropped = bc_image_last_data[crop_start[0]:crop_end[0], crop_start[1]:crop_end[1]]
exp_image_first_cropped = exp_image_first_data[crop_start[0]:crop_end[0], crop_start[1]:crop_end[1]]
exp_image_last_cropped = exp_image_last_data[crop_start[0]:crop_end[0], crop_start[1]:crop_end[1]]

bc_stack = np.stack([bc_image_first_cropped, bc_image_last_cropped])
exp_stack = np.stack([exp_image_first_cropped, exp_image_last_cropped])

# Open Napari viewer
viewer = napari.Viewer()

# Add the first image stack
layer_barcodes = viewer.add_image(np.stack(bc_stack),
                          name='Barcodes',
                          opacity = 1,
                          contrast_limits=(0, 5000))

# Add the second image stack
layer_experiment = viewer.add_image(np.stack(exp_stack),
                          name='Experiment',
                          opacity = 0.5,
                          contrast_limits=(300, 2000),
                          colormap='red')

drift_metadata = {}  # Dictionary to store drift metadata

# Function to record drift after manual alignment
def record_drift(event):
    
    current_fov = viewer.dims.current_step[0]  # Get the current FOV index
    translation = layer_experiment.affine.translate
    drift_metadata[current_fov] = translation#(translation[0], translation[1])  # Store as (dy, dx)
    

last_fov = viewer.dims.current_step[0]  # Store initial FOV

def on_fov_change(event):
    global last_fov
    current_fov = viewer.dims.current_step[0]
    
    if current_fov != last_fov:
        # FOV has changed!
        if current_fov not in drift_metadata:
            layer_experiment.affine.translate = [0., 0., 0.]
        else:
            layer_experiment.affine.translate = drift_metadata[current_fov]

        layer_experiment.refresh()
        last_fov = current_fov  # Update last_fov to current


layer_experiment.events.affine.connect(record_drift)
# layer_experiment.metadata['drift_correction'] = drift_metadata
# viewer.dims.events.point.connect(reset_drift)
viewer.dims.events.current_step.connect(on_fov_change)

napari.run()

In [90]:
print(drift_metadata)

{0: array([   0.        ,  -86.84210635, -121.0526331 ]), 1: array([  0.        , 169.73684424,  67.105264  ])}


### Nd2 reader for experiment data


In [3]:
from nd2reader import ND2Reader
# import h5py
import numpy as np
import os

HEADPATH = '/home/lag36/scratch/lag36/2024-09-05_lLAG8_EZRDM_Run-01/Drift-Correction/'
nd2_filename = HEADPATH + 'Experiment_crop_XY-All_T-Last.nd2'
output_filename = HEADPATH + 'Experiment_crop_XY-All_T-Last_For-Drift.hdf5'

# Define the cropping parameters as variables
crop_x = 1000  # Replace with your desired top-left x-coordinate
crop_y = 600  # Replace with your desired top-left y-coordinate
crop_width = 400  # Replace with your desired width
crop_height = 550 # Replace with your desired height

with ND2Reader(nd2_filename) as nd2file:
    # Get the number of views (positions) from the ND2 file metadata
    num_views = nd2file.sizes['v']
    print(f"Number of views in the ND2 file: {num_views}")
     # Determine the data type from the first frame to preallocate the array
    first_frame = nd2file.get_frame_2D(t=0, v=0)
    dtype = first_frame.dtype
    print(f"Data type of the first frame: {dtype}")

    # Preallocate the 3D NumPy array
    cropped_array_3D = np.empty((num_views, crop_height, crop_width), dtype=dtype)

    # Iterate through each view (since there is only one timepoint: t=0)
    for v in range(num_views):
        # Get the 2D frame for the current view at timepoint 0
        frame = nd2file.get_frame_2D(t=0, v=v)

        # Crop the image
        cropped_array_3D[v, :, :] = frame[crop_y:crop_y + crop_height, crop_x:crop_x + crop_width]

        print(f"Cropped image for view {v} and stored in array.")

Number of views in the ND2 file: 627
Data type of the first frame: uint16
Cropped image for view 0 and stored in array.
Cropped image for view 1 and stored in array.
Cropped image for view 2 and stored in array.
Cropped image for view 3 and stored in array.
Cropped image for view 4 and stored in array.
Cropped image for view 5 and stored in array.
Cropped image for view 6 and stored in array.
Cropped image for view 7 and stored in array.
Cropped image for view 8 and stored in array.
Cropped image for view 9 and stored in array.
Cropped image for view 10 and stored in array.
Cropped image for view 11 and stored in array.
Cropped image for view 12 and stored in array.
Cropped image for view 13 and stored in array.
Cropped image for view 14 and stored in array.
Cropped image for view 15 and stored in array.
Cropped image for view 16 and stored in array.
Cropped image for view 17 and stored in array.
Cropped image for view 18 and stored in array.
Cropped image for view 19 and stored in arr

In [4]:
output_filename

'/home/lag36/scratch/lag36/2024-09-05_lLAG8_EZRDM_Run-01/Drift-Correction/Experiment_crop_XY-All_T-Last_For-Drift.hdf5'

In [5]:
import h5py

In [6]:
# # Open the HDF5 file for writing
with h5py.File(output_filename, 'w') as hf:
    # Create a single dataset in the HDF5 file and write the 3D array
    hf.create_dataset('data', data=cropped_array_3D)
    print(f"Saved the 3D array to dataset 'data' in: {output_filename}")

print("Cropping, stacking, and saving to HDF5 complete.")

Saved the 3D array to dataset 'data' in: /home/lag36/scratch/lag36/2024-09-05_lLAG8_EZRDM_Run-01/Drift-Correction/Experiment_crop_XY-All_T-Last_For-Drift.hdf5
Cropping, stacking, and saving to HDF5 complete.


In [ ]:
positions

[[401 409 379 ... 369 366 415]
 [358 355 375 ... 412 370 406]
 [391 415 373 ... 352 395 394]
 ...
 [424 382 397 ... 375 406 414]
 [376 423 405 ... 392 372 401]
 [439 417 381 ... 365 382 401]]
